# 06 — Поиск аудио ↔ страница

Запустите две первые ячейки один раз. Затем используйте **только одну** из двух независимых ячеек ниже: либо таймкод → предсказанная страница, либо страница → предсказанное аудио. Никаких оценок, ручной разметки и сохранения результатов здесь нет.

Перед запуском должны быть готовы результаты `03c_current_corpus_audio_matching.ipynb`, `05_ocr_text_matching.ipynb` и `05_page_text_map.ipynb`.

In [1]:
from __future__ import annotations

from bisect import bisect_right
import os
import pickle
from pathlib import Path

PROJECT_ROOT = Path(os.environ.get(
    'SPARK_ROOT',
    '/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit',
))
OCR_EXPERIMENT = 'experiment_03_lower_box_threshold'
TEXT_DATA_PATH = PROJECT_ROOT / 'outputs/besy/run_01/normalized_text.pkl'
AUDIO_MAP_PATH = PROJECT_ROOT / 'outputs/besy/run_01/current_corpus_best_result_analysis/audio_map.pkl'
AUDIO_SEGMENTS_PATH = PROJECT_ROOT / 'outputs/besy/run_01/audio_segments.pkl'
PAGE_MAP_PATH = PROJECT_ROOT / 'outputs/besy/page_map' / OCR_EXPERIMENT / 'page_map.pkl'

for path in (TEXT_DATA_PATH, AUDIO_MAP_PATH, AUDIO_SEGMENTS_PATH, PAGE_MAP_PATH):
    assert path.is_file(), f'Не найден обязательный файл: {path}'

with TEXT_DATA_PATH.open('rb') as file:
    book_text = pickle.load(file)['normalized_text']
with AUDIO_MAP_PATH.open('rb') as file:
    audio_map = pickle.load(file)
with AUDIO_SEGMENTS_PATH.open('rb') as file:
    timeline = pickle.load(file)['timeline']
with PAGE_MAP_PATH.open('rb') as file:
    page_map = pickle.load(file)['page_map']

audio_map.sort(key=lambda item: item['audio_start'])
page_map.sort(key=lambda item: item['page_number'])
assert all(right['char_start'] >= left['char_start'] for left, right in zip(audio_map, audio_map[1:]))
assert all(right['char_start'] > left['char_start'] for left, right in zip(page_map, page_map[1:]))

audio_starts = [item['audio_start'] for item in audio_map]
audio_positions = [item['char_start'] for item in audio_map]
page_numbers = [item['page_number'] for item in page_map]
page_positions = [item['char_start'] for item in page_map]
timeline_starts = [item['global_start'] for item in timeline]

print(f'Аудиогрупп: {len(audio_map):,}; диапазон 0–{audio_map[-1]["audio_end"] / 3600:.1f} ч')
print(f'Страниц в карте: {page_numbers[0]}–{page_numbers[-1]}')


Аудиогрупп: 2,088; диапазон 0–36.2 ч
Страниц в карте: 10–699


In [2]:
def parse_time(value: str | float | int) -> float:
    if isinstance(value, (int, float)):
        return float(value)
    parts = [float(part) for part in str(value).strip().split(':')]
    if len(parts) == 1:
        return parts[0]
    if len(parts) == 2:
        return parts[0] * 60 + parts[1]
    if len(parts) == 3:
        return parts[0] * 3600 + parts[1] * 60 + parts[2]
    raise ValueError('Используйте секунды, ММ:СС или ЧЧ:ММ:СС.')


def format_time(seconds: float) -> str:
    hours, seconds = divmod(max(0, round(seconds)), 3600)
    minutes, seconds = divmod(seconds, 60)
    return f'{hours}:{minutes:02}:{seconds:02}'


def text_quote(char_position: int, radius: int = 220) -> str:
    start = max(0, char_position - radius)
    end = min(len(book_text), char_position + radius)
    return ('…' if start else '') + ' '.join(book_text[start:end].split()) + ('…' if end < len(book_text) else '')


def file_and_local_time(global_seconds: float) -> tuple[str, float]:
    index = max(0, bisect_right(timeline_starts, global_seconds) - 1)
    item = timeline[index]
    return item['file'], max(0.0, global_seconds - item['global_start'])


def audio_to_page(global_time: str | float | int) -> dict:
    seconds = parse_time(global_time)
    if not 0 <= seconds <= audio_map[-1]['audio_end']:
        raise ValueError(f'Таймкод вне диапазона 0–{format_time(audio_map[-1]["audio_end"])}.')
    audio_index = max(0, bisect_right(audio_starts, seconds) - 1)
    audio = audio_map[audio_index]
    page_index = max(0, bisect_right(page_positions, audio['char_start']) - 1)
    page = page_map[page_index]
    file_name, local_seconds = file_and_local_time(seconds)
    return {'input_time': format_time(seconds), 'audio': audio, 'page': page, 'file': file_name, 'local_time': format_time(local_seconds)}


def page_to_audio(page_number: int) -> dict:
    page_index = bisect_right(page_numbers, page_number) - 1
    if page_index < 0 or page_numbers[page_index] != page_number:
        raise ValueError(f'Страница вне диапазона {page_numbers[0]}–{page_numbers[-1]}.')
    page = page_map[page_index]
    audio_index = max(0, bisect_right(audio_positions, page['char_start']) - 1)
    audio = audio_map[audio_index]
    file_name, local_seconds = file_and_local_time(audio['audio_start'])
    return {'page': page, 'audio': audio, 'file': file_name, 'local_time': format_time(local_seconds)}


## Таймкод → предсказанная страница

Введите глобальный таймкод в формате `ЧЧ:ММ:СС`. Эта ячейка работает только в направлении аудио → страница.

In [7]:
INPUT_GLOBAL_TIME = '11:59:17'  # ← меняйте только это значение

audio_to_page_result = audio_to_page(INPUT_GLOBAL_TIME)
audio, page = audio_to_page_result['audio'], audio_to_page_result['page']
print(f"Предсказанная страница: {page['page_number']} ({page['source']})")
print(f"Входной глобальный таймкод: {audio_to_page_result['input_time']}")
print(f"Аудиофайл: {audio_to_page_result['file']}")
print(f"Локальный таймкод: {audio_to_page_result['local_time']}; группа: {format_time(audio['audio_start'])}–{format_time(audio['audio_end'])}")
print(f"Уверенность audio matching: {audio['score']:.3f}")
print('\nЦитата:\n' + text_quote(audio['char_start']))


Предсказанная страница: 237 (interpolated)
Входной глобальный таймкод: 11:59:17
Аудиофайл: 201 Ночь.mp3
Локальный таймкод: 1:31:46; группа: 11:58:51–11:59:51
Уверенность audio matching: 0.823

Цитата:
…смотря на ставрогина. и ударили? шатов вспыхнул и забормотал почти без связи: я за ваше падение за ложь. я не для того подходил, чтобы вас наказать; когда я подходил, я не знал, что ударю я за то, что вы так много значили в моей жизни я понимаю, понимаю, берегите слова. мне жаль, что вы в жару; у меня самое необходимое дело. я слишком долго вас ждал, как-то весь чуть не затрясся шатов и привстал было с места, говорите ваше дело, я тоже…


## Страница → предсказанное аудио

Введите номер физической страницы. Эта ячейка работает только в направлении страница → аудио.

In [8]:
INPUT_PAGE_NUMBER = 408  # ← меняйте только это значение

page_to_audio_result = page_to_audio(INPUT_PAGE_NUMBER)
audio, page = page_to_audio_result['audio'], page_to_audio_result['page']
print(f"Входная страница: {page['page_number']} ({page['source']})")
print(f"Предсказанное аудио: {page_to_audio_result['file']}")
print(f"Глобальный таймкод: {format_time(audio['audio_start'])}; локальный: {page_to_audio_result['local_time']}")
print(f"Группа: {format_time(audio['audio_start'])}–{format_time(audio['audio_end'])}")
print(f"Уверенность audio matching: {audio['score']:.3f}")
print('\nЦитата:\n' + text_quote(page['char_start']))


Входная страница: 408 (interpolated)
Предсказанное аудио: 208 Иван-Царевич.mp3
Глобальный таймкод: 20:53:18; локальный: 0:17:07
Группа: 20:53:18–20:54:20
Уверенность audio matching: 0.812

Цитата:
…одойдет вас потрепать по плечу. вы ужасный аристократ. аристократ, когда идет в демократию, обаятелен! вам ничего не значит пожертовать жизнью, и своею и чужою. вы именно таков, какого надо. мне, мне именно такого надо, как вы. я ни кого, кроме вас, не знаю. вы предводитель, вы солнце, а я ваш червяк он вдруг поцеловал у него руку. холод прошел по спине ставрогина, и он в испуге вырвал свою руку. они остановились. помешанный! прошептал…
